# 春学期第 1 回｜データ分析と Python

迎えに来る人に「定刻より早く着く便はある？」と聞かれたら，どの記録を見せればよいでしょうか。2024 年 1 月の便の図で早着の有無を読み，空港や日付を絞っても同じ特徴が見えるかを調べます。元の表，図，件数をつないで答えましょう。

最初に本文の例を一緒に実行し，後半では自分で条件を選んで約 20 分の実習に取り組みます。

## 準備

Colab では「ドライブにコピーを保存」で自分用のノートを作り，上から準備セルを実行します。CSV は公開 URL から読み込むため，アップロードは不要です。ローカルでは，実習 ZIP を展開してノートブックと CSV を同じフォルダーに置くと，手元のファイルを使えます。

最初のセルは配布場所の違いに対応する準備用です。そのまま実行してください。出力の `32347` は，この月に記録された予定便の行数です。図に描ける到着のずれがある便数とは区別します。読み込みの詳細は Book の「Python と学習環境」にあります。

In [ ]:
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt

file_name = "nyc-flights-2024-01.csv"
csv_path = Path("data") / file_name
if not csv_path.is_file():
    csv_path = Path("../data") / file_name
if not csv_path.is_file():
    csv_path = Path(file_name)
if not csv_path.is_file():
    csv_path = "https://raw.githubusercontent.com/sohei-takagi/python-ds-course-materials/main/data/" + file_name

flights = pd.read_csv(csv_path)
len(flights)

## 本文と一緒に実行

各節で何を確かめるかを読んでから，コードを上から順に実行します。図を描くセルでは，横軸が何分を表すか，棒の高さが何を数えているかを出力と対応させます。

### 定刻より早く着く便と遅く着く便

2024 年 1 月の 3 空港の便を合わせた図から始めます。破線の 0 分を基準に，早着側に便があるか，遅着側の棒がどこまで続くかを読みます。一番右の便が，この月の多くの便を代表しているかも棒の高さで確かめます。

In [ ]:
# 提供した作図セル。そのまま実行します。
import matplotlib.pyplot as plt
import pandas as pd

overview = flights
overview_delays = overview["arrival_delay"].dropna()

fig, ax = plt.subplots(layout="constrained")
ax.hist(overview_delays, bins=162, range=(-90, 1530), color="#286b70")
ax.axvline(0, color="#603c32", linestyle="--", linewidth=1.2)
ax.annotate("1,500 min", xy=(1500, 0), xytext=(1470, 1250), ha="right",
            color="#603c32", arrowprops={"arrowstyle": "->", "color": "#603c32"})
ax.set(xlim=(-108, 1540), xticks=[-90, 0, 300, 600, 900, 1200, 1500],
       xlabel="Arrival delay (min)", ylabel="Flights")
ax.spines[["top", "right"]].set_visible(False)
plt.show()

同じ便のうち，−60 分から 120 分までを拡大します。最も多くの便が入る 10 分幅の区間と，定刻の 0 分を挟む左右の棒を読みます。全範囲の図で見えにくかった早着側の集まりが分かるでしょうか。

In [ ]:
# 提供した作図セル。そのまま実行します。
fig, ax = plt.subplots(layout="constrained")
ax.hist(overview_delays, bins=18, range=(-60, 120), color="#286b70", edgecolor="white")
ax.axvline(0, color="#603c32", linestyle="--", linewidth=1.2)
ax.set(xlim=(-60, 120), xticks=[-60, -30, 0, 30, 60, 90, 120],
       xlabel="Arrival delay (min)", ylabel="Flights")
ax.spines[["top", "right"]].set_visible(False)
plt.show()

### 出発空港ごとに描く

全空港を合わせた図の特徴が，ニューアーク（EWR）発だけでも見えるか調べます。まず出発空港で行を選び，全予定便，EWR の予定便，到着のずれが記録された EWR の便の順に件数を確認します。最後の件数が図へ渡す値の個数です。

In [ ]:
import pandas as pd

flights.head(3)

In [ ]:
airport = "EWR"

In [ ]:
selected_flights = flights.loc[flights["origin"] == airport]
selected_flights.head(3)

In [ ]:
arrival_delays = selected_flights["arrival_delay"].dropna()
print(len(flights), len(selected_flights), len(arrival_delays))

EWR とジョン・F・ケネディ（JFK）の図を，同じ横軸の範囲と 10 分幅の棒で描きます。それぞれ最も高い棒がどの区間にあるかを比べます。棒の高さは便数なので，空港ごとの観測便数も一緒に読んでください。

In [ ]:
# 提供した作図セル。そのまま実行します。
import matplotlib.pyplot as plt

fig, ax = plt.subplots(layout="constrained")
ax.hist(arrival_delays, bins=18, range=(-60, 120), color="#286b70", edgecolor="white")
ax.axvline(0, color="#603c32", linestyle="--")
ax.set(xlim=(-60, 120), ylim=(0, 2600), title=airport,
       xlabel="Arrival delay (min)", ylabel="Flights")
plt.show()

In [ ]:
# 提供した作図セル。そのまま実行します。
import matplotlib.pyplot as plt

comparison_airport = "JFK"
comparison_flights = flights.loc[flights["origin"] == comparison_airport]
comparison_delays = comparison_flights["arrival_delay"].dropna()

fig, ax = plt.subplots(layout="constrained")
ax.hist(comparison_delays, bins=18, range=(-60, 120), color="#286b70", edgecolor="white")
ax.axvline(0, color="#603c32", linestyle="--")
ax.set(xlim=(-60, 120), ylim=(0, 2600), title=comparison_airport,
       xlabel="Arrival delay (min)", ylabel="Flights")
plt.show()

### 表の 1 行が図の棒に入るまで

元の表の先頭 6 便だけを使い，棒の高さの作り方を追います。上段の各点は 1 便の値です。−10 分の便と −16 分の便は，下段のどの区間に入り，それぞれ棒を何便分高くしますか。

In [ ]:
six_flights = flights.head(6)
six_flights[["date", "origin", "destination", "scheduled_departure", "arrival_delay"]]

In [ ]:
# 提供した作図セル。そのまま実行します。
fig, (rows_ax, histogram_ax) = plt.subplots(2, 1, sharex=True, layout="constrained")
six_delays = six_flights["arrival_delay"]
rows_ax.scatter(six_delays, range(1, 7), s=52, color="#286b70", zorder=3)
rows_ax.set(yticks=range(1, 7), ylabel="Row within excerpt")
rows_ax.invert_yaxis()
histogram_ax.hist(six_delays, bins=4, range=(-40, 0), color="#286b70", edgecolor="white")
histogram_ax.set(xlim=(-40, 0), xticks=[-40, -30, -20, -10, 0], yticks=[0, 1, 2],
                 ylim=(0, 2.4), xlabel="Arrival delay (min)", ylabel="Flights")
for panel in (rows_ax, histogram_ax):
    panel.grid(axis="x", alpha=0.35, linestyle="--")
    panel.spines[["top", "right"]].set_visible(False)
plt.show()

先頭 3 行だけを取り出した表と，CSV から読んだ全表の行数を比べます。画面に見える行が 3 行でも，全表から空港や日付で便を選ぶときは，4 行目以降も候補に残っています。

In [ ]:
preview = flights.head(3)
print(len(preview), len(flights))

### 到着と迎えの時刻を計算する

到着予定時刻を 0 分の基準にします。到着が −10 分，迎えが +10 分なら，迎えまで何分ですか。次に到着が +18 分になったとき，差の符号を読みます。

In [ ]:
# 提供した作図セル。そのまま実行します。
fig, ax = plt.subplots(layout="constrained")
ax.hlines([0, 1], -21, 28, color="#b5b5b5", linewidth=1)
ax.vlines(0, -0.48, 1.22, color="#603c32", linestyle="--", linewidth=1.2)
ax.vlines(10, -0.48, 1.22, color="#676767", linestyle=":", linewidth=1.2)
ax.annotate("", xy=(10, 1), xytext=(-10, 1),
            arrowprops={"arrowstyle": "->", "color": "#286b70", "lw": 1.5})
ax.annotate("", xy=(10, 0), xytext=(18, 0),
            arrowprops={"arrowstyle": "->", "color": "#286b70", "lw": 1.5})
ax.scatter([-10, 18], [1, 0], marker="o", color="#286b70", s=56, zorder=3)
ax.scatter([10, 10], [1, 0], marker="s", color="#b15334", s=52, zorder=3)
ax.text(0, 1.35, "Scheduled arrival", ha="center")
ax.text(10, 1.35, "Pickup plan", ha="center")
ax.set(xlim=(-22, 29), ylim=(-0.48, 1.64), xticks=[-10, 0, 10, 18],
       yticks=[1, 0], yticklabels=["EWR–ATL", "EWR–FLL"],
       xlabel="Minutes relative to scheduled arrival")
ax.spines[["top", "left", "right"]].set_visible(False)
ax.tick_params(axis="y", length=0)
plt.show()

In [ ]:
arrival_delay = -10
pickup_offset = 10
gap = pickup_offset - arrival_delay
gap

In [ ]:
arrival_delay = 18
gap = pickup_offset - arrival_delay
gap

In [ ]:
pickup_offset = 25
gap = pickup_offset - arrival_delay
print(pickup_offset, arrival_delay, gap)

到着の値を +18 分に代入し直しても，先に計算した差はそのままです。最後の行がどの時点の値を表示するか，実行前に予想します。

In [ ]:
pickup_offset = 10
arrival_delay = -10
gap = pickup_offset - arrival_delay
arrival_delay = 18
gap

### 空港を選び直す

EWR の表を作った後で，空港の指定だけを JFK に替えます。この時点で表示した空港名と，選択済みの表の `origin` は一致するでしょうか。古い表が残る理由を確かめてから，条件抽出と到着のずれの取り出しを順に実行し直します。

In [ ]:
airport = "EWR"
selected_flights = flights.loc[flights["origin"] == airport]
arrival_delays = selected_flights["arrival_delay"].dropna()

In [ ]:
airport = "JFK"
print(airport, len(selected_flights), len(arrival_delays))
selected_flights["origin"].head(1)

In [ ]:
selected_flights = flights.loc[flights["origin"] == airport]
arrival_delays = selected_flights["arrival_delay"].dropna()
print(airport, len(selected_flights), len(arrival_delays))

### 日付を選び，遅く着いた便を数える

月全体の図だけでは，日ごとの違いは分かりません。1 月 1 日の便を選び，同じ軸で 16 日と比べます。最も高い棒は各日で 0 分のどちら側にあり，遅着側の棒はどちらの日に長く続きますか。

In [ ]:
day = "2024-01-01"
day_flights = flights.loc[flights["date"] == day]
day_delays = day_flights["arrival_delay"].dropna()
print(day, len(day_flights), len(day_delays))

In [ ]:
# 提供した作図セル。そのまま実行します。
import matplotlib.pyplot as plt

day1_flights = flights.loc[flights["date"] == "2024-01-01"]
day16_flights = flights.loc[flights["date"] == "2024-01-16"]
day1_delays = day1_flights["arrival_delay"].dropna()
day16_delays = day16_flights["arrival_delay"].dropna()

fig, (day1_ax, day16_ax) = plt.subplots(2, 1, sharex=True, sharey=True, layout="constrained")
day1_ax.hist(day1_delays, bins=42, range=(-60, 360), color="#286b70", edgecolor="white")
day16_ax.hist(day16_delays, bins=42, range=(-60, 360), color="#286b70", edgecolor="white")
day1_ax.set(title="January 1", ylabel="Flights")
day16_ax.set(title="January 16", xlim=(-60, 360), ylim=(0, 180),
             xticks=[-60, 0, 60, 120, 180, 240, 300, 360],
             xlabel="Arrival delay (min)", ylabel="Flights")
day1_ax.axvline(0, color="#603c32", linestyle="--", linewidth=1.2)
day16_ax.axvline(0, color="#603c32", linestyle="--", linewidth=1.2)
day1_ax.spines[["top", "right"]].set_visible(False)
day16_ax.spines[["top", "right"]].set_visible(False)
plt.show()

1 月 16 日の図では遅着側に棒が続きます。0 分より大きい値に `True` を付けて足し，遅く着いた便を数えます。図の表示範囲外にある便も判定に入ることと，何便の到着記録から数えたかを確かめます。

In [ ]:
day = "2024-01-16"
day_flights = flights.loc[flights["date"] == day]
day_delays = day_flights["arrival_delay"].dropna()
day_delays.head(6)

In [ ]:
is_late_on_day = day_delays > 0
is_late_on_day.head(6)

In [ ]:
late_count_on_day = is_late_on_day.sum()
print(late_count_on_day, len(day_delays))

## Jupyter 実習（約 20 分）

図を見て気になったことを一つ選び，別の空港でも同じ特徴が見られるかを調べます。空港名を変えた図だけでなく，**自分の問いと，調べて分かったこと**を残してください。

共通課題は 1〜3 の約 20 分です。末尾の日付比較は任意の発展で，共通課題の提出には必要ありません。

### 1. 図から問いを選ぶ（4 分）

まず基準として EWR 発の図を表示します。以下の三つのコードセルを順に実行してください。この基準図は残し，後で作るもう一つの空港の図と見比べます。

In [ ]:
airport = "EWR"
selected_flights = flights.loc[flights["origin"] == airport]
selected_flights.head(3)

In [ ]:
arrival_delays = selected_flights["arrival_delay"].dropna()
print("予定便:", len(selected_flights))
print("遅れが記録された便:", len(arrival_delays))

In [ ]:
fig, ax = plt.subplots()
ax.hist(arrival_delays, bins=18, range=(-60, 120))
ax.axvline(0, color="black", linestyle="--")
ax.set(xlim=(-60, 120), ylim=(0, 2600), title=airport,
       xlabel="Arrival delay (minutes)", ylabel="Flights")
plt.show()

図を見て，別の空港でも調べたいことを一つ決めます。例えば，次のどちらかです。

- 最も高い棒は，EWR と同じ到着のずれの区間にあるだろうか。
- 0 分より左に高い棒が集まる特徴は，他の空港でも見られるだろうか。

講義で既に見た特徴でも，自分が選ぶ空港について，どの棒や位置を確かめるかを決めてください。思い付いた別の問いでも，今ある列と図で調べられるものなら構いません。

**自分の問い：**

**比較する空港（JFK または LGA）：**

**調べる前の予想と，図のどこを見て確かめるか：**

### 2. 自分で比較する（10 分）

下の空港名を，自分が選んだものにします。そのセルから図までを順に実行してください。比較する二つの図は区間の幅と軸をそろえています。色や装飾の変更はこの実習では必要ありません。

In [ ]:
airport = "JFK"

In [ ]:
selected_flights = flights.loc[flights["origin"] == airport]
selected_flights.head(3)

In [ ]:
arrival_delays = selected_flights["arrival_delay"].dropna()
print("予定便:", len(selected_flights))
print("遅れが記録された便:", len(arrival_delays))

In [ ]:
fig, ax = plt.subplots()
ax.hist(arrival_delays, bins=18, range=(-60, 120))
ax.axvline(0, color="black", linestyle="--")
ax.set(xlim=(-60, 120), ylim=(0, 2600), title=airport,
       xlabel="Arrival delay (minutes)", ylabel="Flights")
plt.show()

**選んだ空港の予定便数：**

**到着の遅れが記録された便数：**

**EWR の図と同じだった特徴：**

**違っていた特徴，または判断するには情報が足りなかった点：**

同じ特徴が見つかったことも分析の結果です。違いを無理に見つける必要はありません。棒の高さの違いを書くときは，空港ごとに便数が異なることも説明に含めます。

#### 図の特徴を件数でも調べる

選んだ空港では，定刻より遅く着いた便が何便あったでしょうか。本文で 1 月 16 日の便を数えた手順を読み替え，今回は選んだ空港の `arrival_delays` を判定します。

次のセルのコメントを参考に，`is_late` に判定を入れる一行を書いてから実行してください。表示する行 `# is_late.head(6)` の先頭の `#` も外します。`#` で始まる行はコメントとして扱われ，そのままでは実行されません。

見本の `day_delays` をそのまま使うと，別の対象を数えてしまいます。

書き方が分からなければ本文の「遅く着いた便を数える」を参照して構いません。先に自分で書き，出力が何を数えたかを説明することが課題です。

In [ ]:
# arrival_delays の値が 0 より大きいかを比べ，is_late に入れる

# 上の一行を書いた後で，先頭の判定を表示する
# is_late.head(6)

In [ ]:
# 判定を作ったら，次の2行の先頭の # を外して実行する
# late_count = is_late.sum()
# print(late_count, len(arrival_delays))

**遅く着いた便数 ／ 到着の遅れが記録された便数：**

**図のどの部分に入る便を数えたか（範囲外の便も含むか）：**

<details>
<summary>確認用のコード</summary>

```python
is_late = arrival_delays > 0
is_late.head(6)
late_count = is_late.sum()
print(late_count, len(arrival_delays))
```

JFK なら `3239 9177`，LGA なら `3862 12086` になります。正しい件数だけでなく，判定を足すと便数になる理由も説明してください。
</details>

### 3. 発見を説明する（6 分）

最初に書いた問いへ戻り，次の三点を短い文章でまとめてください。

1. **分かったこと。** 2024 年 1 月のどの空港の記録を比べ，何を見つけたか。
2. **図と件数の根拠。** 横軸の区間や位置を具体的に挙げる。例えば「左が高い」ではなく，「−20 分以上 −10 分未満の区間に最も多く集まる」のように書く。遅く着いた便数は，到着の遅れが記録された便数と一緒に示す。
3. **次に調べたいこと。** 日によって違うか，特定の行先でも見られるかなど。今の結果からその疑問が出た理由も一文添える。

**説明：**

**次の問い：**

二つの空港で棒の高さが違っても，その区間に入った便の件数と，図に渡した便の総数を区別して説明してください。

提出前に全セルを上から再実行します。上の EWR の基準図と，下の自分が選んだ空港の図が両方残り，本文の空港名と件数が対応していることを確かめて保存してください。提出物はこのノートです。

### 任意の発展：別の日も調べる

共通課題が終わった人は，日付を変えて図を調べられます。講義で見た 1 月 1 日・16 日のほか，この月の別の日を一つ選んでください。ここでは 3 空港を合わせ，空港比較とは別に日付を条件にします。

日ごとの図は，講義と同じ 10 分幅で −60〜360 分を描きます。120 分より大きい遅れも見えるようにするためです。この範囲の外にある値は図には入りません。縦軸は，この月のどの日を選んでも棒が収まる 350 便までとしています。講義の 2 日だけの比較図とは縦軸の上限が異なるので，棒の画面上の高さを直接比べず，軸の数値も読んでください。コードを実行する前に，どんな形になりそうか一言書いてみましょう。

**選んだ日付と，図についての予想：**

In [ ]:
day = "2024-01-08"
day_flights = flights.loc[flights["date"] == day]
day_delays = day_flights["arrival_delay"].dropna()
print("予定便:", len(day_flights))
print("遅れが記録された便:", len(day_delays))

In [ ]:
fig, ax = plt.subplots()
ax.hist(day_delays, bins=42, range=(-60, 360))
ax.axvline(0, color="black", linestyle="--")
ax.set(xlim=(-60, 360), ylim=(0, 350), title=day,
       xlabel="Arrival delay (minutes)", ylabel="Flights")
plt.show()

**講義の 2 日の図と比べて気付いたこと：**

**この違いをさらに調べるなら，どんな列や資料が欲しいか：**

選んだ日の図だけから，遅れの原因や今後の便の到着時刻まで分かったとは言えません。まず図の違いを記述し，考えられる説明と，それを調べるための情報を分けて書きましょう。